# NexusQuant — KV Cache Compression Demo

**Compress your LLM's KV cache 10–33x. Training-free. One line of code.**

This notebook shows:
1. Installing NexusQuant
2. Loading TinyLlama on CPU (no GPU required)
3. Baseline generation (no compression)
4. Compressed generation with `high`, `balanced`, and `max` presets
5. Compression statistics

Runs in under 2 minutes on Colab free tier.

---
| Preset | Compression | PPL degradation |
|--------|------------|----------------|
| `high` | 10x | +0.4% |
| `balanced` | 17x | +1.3% |
| `max` | 33x | +2.6% |

*Measured on Mistral-7B, A100, FP16. All ratios include overhead (scales, indices, metadata).*

## 1. Install

In [ ]:
!pip install -q nexusquant-kv "nexusquant-kv[hf]" transformers accelerate

## 2. Load TinyLlama on CPU

We use `TinyLlama-1.1B-Chat-v1.0` — small enough to run on Colab free tier in under a minute.

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model and tokenizer...")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float32,   # CPU: use fp32
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

## 3. Prepare the prompt

NexusQuant's importance scorer needs at least ~500 tokens of context to work well.
We repeat a paragraph to give it enough signal.

In [ ]:
PARAGRAPH = (
    "Large language models have transformed natural language processing. "
    "Attention mechanisms enable models to weigh the relevance of each token "
    "when generating the next one. As context lengths grow, the key-value cache "
    "that stores intermediate attention state becomes the dominant memory cost. "
    "NexusQuant reduces that cost at inference time without any training. "
)
prompt = PARAGRAPH * 8   # ~640 tokens

input_ids = tok(prompt, return_tensors="pt").input_ids
print(f"Prompt length: {input_ids.shape[1]} tokens")

## 4. Baseline generation (no compression)

In [ ]:
print("Running baseline (no compression)...")
t0 = time.time()

with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=60, do_sample=False)

baseline_time = time.time() - t0
baseline_text = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

print(f"Time: {baseline_time:.1f}s")
print(f"\nBaseline output:\n  {baseline_text.strip()}")

## 5. Compressed generation — all three presets

Each `with nexusquant_evict(model, quality=...)` block:
1. Installs compression hooks on the model
2. Runs generation with compressed KV cache
3. Automatically removes hooks when the block exits (model is unchanged)

No model modification, no training, no calibration data.

In [ ]:
from nexusquant import nexusquant_evict

PRESETS = [
    ("high",     "10x",  "+0.4% PPL"),
    ("balanced", "17x",  "+1.3% PPL"),
    ("max",      "33x",  "+2.6% PPL"),
]

results = []

for preset, compression, quality_note in PRESETS:
    print(f"\n--- Preset: {preset!r}  ({compression} compression, {quality_note}) ---")
    t0 = time.time()

    with nexusquant_evict(model, quality=preset):
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=60, do_sample=False)

    elapsed = time.time() - t0
    text = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    results.append((preset, compression, quality_note, elapsed, text))

    print(f"Time: {elapsed:.1f}s")
    print(f"Output: {text.strip()}")

print("\n--- All presets done ---")

## 6. Compression stats summary

In [ ]:
# KV cache size estimate: 2 (K+V) * layers * heads * head_dim * seq_len * bytes_per_element
config = model.config
layers      = config.num_hidden_layers
heads       = config.num_key_value_heads if hasattr(config, 'num_key_value_heads') else config.num_attention_heads
head_dim    = config.hidden_size // config.num_attention_heads
seq_len     = input_ids.shape[1]
bytes_fp32  = 2 * layers * heads * head_dim * seq_len * 4   # 4 bytes/float (fp32)
bytes_fp16  = bytes_fp32 / 2

print("=" * 62)
print(f"  Model             : TinyLlama-1.1B-Chat")
print(f"  Layers            : {layers}")
print(f"  KV heads          : {heads}")
print(f"  Head dim          : {head_dim}")
print(f"  Prompt tokens     : {seq_len}")
print(f"  Baseline KV (fp16): {bytes_fp16/1024:.1f} KB  ({bytes_fp16/1e6:.2f} MB)")
print("=" * 62)

RATIOS = {"high": 10, "balanced": 17, "max": 33}

print(f"  {'Preset':<10}  {'Compression':>12}  {'KV (fp16)':>12}  {'Compressed':>12}  {'Saved':>8}")
print("  " + "-" * 58)

for preset, compression, quality_note, elapsed, _ in results:
    ratio = RATIOS[preset]
    compressed_bytes = bytes_fp16 / ratio
    saved_pct = (1 - 1/ratio) * 100
    print(
        f"  {preset:<10}  {compression:>12}  "
        f"{bytes_fp16/1024:>9.1f} KB  "
        f"{compressed_bytes/1024:>9.1f} KB  "
        f"{saved_pct:>6.0f}%"
    )

print("=" * 62)
print()
print("Note: these are theoretical KV cache sizes for the prompt.")
print("Actual GPU savings require native CUDA compact storage kernels.")
print("The current implementation is a CPU-friendly reference.")

## 7. What this means at scale

These compression ratios unlock much longer contexts on the same hardware:

| Preset | Compression | 128K context → | Context on 80 GB A100 |
|--------|------------|---------------|-----------------------|
| `high` | 10x | 1.28M tokens | ~1.3M tokens |
| `balanced` | 17x | 2.18M tokens | ~2.2M tokens |
| `max` | 33x | 4.22M tokens | ~4.2M tokens |

## Next steps

```python
# Drop-in for any HuggingFace causal LM
from nexusquant import nexusquant_evict

with nexusquant_evict(model, quality="balanced"):
    output = model.generate(input_ids, max_new_tokens=512)
```

- **Install:** `pip install nexusquant-kv`  
- **GitHub:** https://github.com/jagmarques/nexusquant  
- **Supported models:** Llama, Mistral, Mixtral, Qwen, Phi, Gemma  
- **Not yet supported:** models with interleaved RoPE (GPT-NeoX, GPT-J)